In [1]:
import os
import re
import numpy as np
import pandas as pd
import nltk
import tensorflow as tf


In [3]:
from google.colab import files

uploaded = files.upload()

path = list(uploaded.keys())[0]

print("Dataset loaded:", path)

KeyboardInterrupt: 

In [4]:
with open(r"/content/NEXT WORD PREDICTION DATASET ASSIGNENT.txt", 'r', encoding='utf-8-sig') as file:
    text = file.read()

print(text[:1000])


Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: The Adventures of Sherlock Holmes

Author: Arthur Conan Doyle

Release Date: November 29, 2002 [EBook #1661]
Last Updated: May 20, 2019

Language: English

Character set encoding: UTF-8

*** START OF THIS PROJECT GUTENBERG EBOOK THE ADVENTURES OF SHERLOCK HOLMES ***



Produced by an anonymous Project Gutenberg volunteer and Jose Menendez



cover



The Adventures of Sherlock Holmes



by Arthur Conan Doyle



Contents


   I.     A Scandal in Bohemia
   II.    The Red-Headed League
   III.   A Case of Identity
   IV.    The Boscombe Valley Mystery
   V.     The Five Orange Pips
   VI.    The Man with the Twisted Lip
   VII.   The Adventure of the Blue C

In [5]:
start = "*** START OF THIS PROJECT GUTENBERG EBOOK"
end = "*** END OF THIS PROJECT GUTENBERG EBOOK"

text = text.split(start, 1)[1]
text = text.split(end, 1)[0]

text = text.lower()

print(text[:1000])

 the adventures of sherlock holmes ***



produced by an anonymous project gutenberg volunteer and jose menendez



cover



the adventures of sherlock holmes



by arthur conan doyle



contents


   i.     a scandal in bohemia
   ii.    the red-headed league
   iii.   a case of identity
   iv.    the boscombe valley mystery
   v.     the five orange pips
   vi.    the man with the twisted lip
   vii.   the adventure of the blue carbuncle
   viii.  the adventure of the speckled band
   ix.    the adventure of the engineer’s thumb
   x.     the adventure of the noble bachelor
   xi.    the adventure of the beryl coronet
   xii.   the adventure of the copper beeches



i. a scandal in bohemia


i.

to sherlock holmes she is always _the_ woman. i have seldom heard him
mention her under any other name. in his eyes she eclipses and
predominates the whole of her sex. it was not that he felt any emotion
akin to love for irene adler. all emotions, and that one particularly,
were abhorrent to 

In [6]:
nltk.download('punkt_tab')

from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(text)

print("Total sentences:", len(sentences))
print(sentences[:5])

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Total sentences: 4697
[' the adventures of sherlock holmes ***\n\n\n\nproduced by an anonymous project gutenberg volunteer and jose menendez\n\n\n\ncover\n\n\n\nthe adventures of sherlock holmes\n\n\n\nby arthur conan doyle\n\n\n\ncontents\n\n\n   i.     a scandal in bohemia\n   ii.', 'the red-headed league\n   iii.', 'a case of identity\n   iv.', 'the boscombe valley mystery\n   v.     the five orange pips\n   vi.', 'the man with the twisted lip\n   vii.']


In [7]:
clean_sentences = []

for sentence in sentences:

    sentence = re.sub(r'[^a-z\s]', ' ', sentence)

    sentence = re.sub(r'\s+', ' ', sentence).strip()

    if sentence:
        clean_sentences.append(sentence)

print(clean_sentences[:5])

['the adventures of sherlock holmes produced by an anonymous project gutenberg volunteer and jose menendez cover the adventures of sherlock holmes by arthur conan doyle contents i a scandal in bohemia ii', 'the red headed league iii', 'a case of identity iv', 'the boscombe valley mystery v the five orange pips vi', 'the man with the twisted lip vii']


In [8]:
nltk.download('wordnet')

from nltk.stem import WordNetLemmatizer

lema = WordNetLemmatizer()

lem_sentences = []

for sentence in clean_sentences:

    lem_sentence = ' '.join([
        lema.lemmatize(word)
        for word in sentence.split()
    ])

    lem_sentences.append(lem_sentence)

print(lem_sentences[:5])

[nltk_data] Downloading package wordnet to /root/nltk_data...


['the adventure of sherlock holmes produced by an anonymous project gutenberg volunteer and jose menendez cover the adventure of sherlock holmes by arthur conan doyle content i a scandal in bohemia ii', 'the red headed league iii', 'a case of identity iv', 'the boscombe valley mystery v the five orange pip vi', 'the man with the twisted lip vii']


In [9]:
simple_text = ' '.join(clean_sentences)
lem_text = ' '.join(lem_sentences)

text_len = len(simple_text)
lemtextlen = len(lem_text)

print(f'''
Length of simple text = {text_len}
Length of lemmatized text = {lemtextlen}
''')


Length of simple text = 537525
Length of lemmatized text = 531374



In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer

tok = Tokenizer()

tok.fit_on_texts(lem_sentences)

seq = tok.texts_to_sequences(lem_sentences)

tok_word_seq = tok.word_index

print("Vocabulary size:", len(tok_word_seq))

print("First sequence:", seq[0])

Vocabulary size: 7130
First sequence: [1, 430, 6, 131, 32, 2374, 47, 50, 4106, 2966, 2967, 4107, 4, 4108, 4109, 1524, 1, 430, 6, 131, 32, 47, 514, 2968, 2969, 1737, 3, 2, 791, 8, 892, 2375]


In [11]:
data = []

max_sequence_len = 30

for s in seq:

    for i in range(1, len(s)):

        ngram = s[max(0, i-max_sequence_len+1):i+1]

        data.append(ngram)

print("Total sequences:", len(data))

print(data[:5])

Total sequences: 101194
[[1, 430], [1, 430, 6], [1, 430, 6, 131], [1, 430, 6, 131, 32], [1, 430, 6, 131, 32, 2374]]


In [33]:
x = []
y_arr = []

for r in data:
    x.append(r[:-1])
    y_arr.append(r[-1])

x = pad_sequences(
    x,
    maxlen=max_len,
    padding='post'
)

x = np.array(x, dtype=np.int32)
y_arr = np.array(y_arr, dtype=np.int32)

print("X shape:", x.shape)
print("Y shape:", y_arr.shape)

X shape: (101194, 29)
Y shape: (101194,)


In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y_arr,
    test_size=0.23,
    random_state=42
)

In [35]:
from tensorflow.keras import Sequential, layers

vocab_size = len(tok_word_seq) + 1

model = Sequential([
    layers.Input(shape=(max_len,)),

    layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,
        mask_zero=False
    ),

    layers.LSTM(128, return_sequences=True),

    layers.LSTM(64),

    layers.Dense(vocab_size, activation='softmax')
])

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 29, 128)        │       912,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 29, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7131)           │       463,515 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,557,275 (5.94 MB)

 Trainable params: 1,557,275 (5.94 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [37]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=128
)

Epoch 1/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.0524 - loss: 6.6115 - val_accuracy: 0.0546 - val_loss: 6.4989
Epoch 2/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.0526 - loss: 6.3885 - val_accuracy: 0.0546 - val_loss: 6.4829
Epoch 3/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.0534 - loss: 6.2504 - val_accuracy: 0.0581 - val_loss: 6.3343
Epoch 4/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.0589 - loss: 6.0715 - val_accuracy: 0.0633 - val_loss: 6.2396
Epoch 5/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.0641 - loss: 5.9423 - val_accuracy: 0.0659 - val_loss: 6.1852
Epoch 6/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.0717 - loss: 5.8317 - val_accuracy: 0.0767 - val_loss: 6.1089
Epoch 7/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.0890 - loss: 5.6860 - val_accuracy: 0.0900 - val_loss: 6.0369
Epoch 8/10
609/609 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.1043 - loss: 5.5426 - val_a

In [39]:
def predict_next_word(sentence):

    sentence = sentence.lower()

    sentence = re.sub(r'[^a-z\s]', ' ', sentence)

    sentence = re.sub(r'\s+', ' ', sentence).strip()

    sentence = ' '.join([
        lema.lemmatize(word)
        for word in sentence.split()
    ])

    token_list = tok.texts_to_sequences([sentence])

    if not token_list[0]:
        return "No known words found!"

    token_list = token_list[0]

    token_list = pad_sequences(
        [token_list],
        maxlen=max_len,
        padding='pre'
    )

    prediction = model.predict(token_list, verbose=0)

    predicted_index = np.argmax(prediction)

    predicted_word = tok.index_word.get(
        predicted_index,
        "Unknown"
    )

    return predicted_word

In [40]:
print(predict_next_word("sherlock holmes was"))

print(predict_next_word("the man was"))

print(predict_next_word("i have seen"))

that
a
to
